# From LDA to BERTopic: Improving Topic Modeling on the 20 Newsgroups Dataset

## Project Overview

This project extends a previous LDA-based topic modeling project by adding:

1. a cleaner and reproducible baseline LDA pipeline;
2. ground-truth category evaluation using the 20 Newsgroups labels;
3. a BERTopic model as a transformer-based extension;
4. a comparison between LDA and BERTopic using topic keywords, category alignment, and clustering metrics.

**Research question:** Can transformer-based topic modeling improve topic discovery compared with traditional LDA on the 20 Newsgroups dataset?

## Stage 0: Environment Setup

Run this cell first in Google Colab. If the runtime restarts after installing packages, run the notebook again from this point.

In [ ]:
!pip -q install gensim spacy nltk pyLDAvis bertopic sentence-transformers umap-learn hdbscan
!python -m spacy download en_core_web_sm -q

In [ ]:
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
RANDOM_STATE = 100

sns.set_theme(style="whitegrid")

## Stage 1: Load the Dataset

This version uses `fetch_20newsgroups` instead of a local `.tar.gz` file, so the notebook is easier to reproduce in Colab.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes"),
)

docs_raw = dataset.data
labels = dataset.target
target_names = dataset.target_names
label_names = [target_names[i] for i in labels]

print(f"Number of documents: {len(docs_raw)}")
print(f"Number of categories: {len(target_names)}")
print(target_names)

## Stage 2: Text Cleaning and Preprocessing for LDA

LDA is a bag-of-words model, so it benefits from stronger preprocessing: cleaning, lemmatization, stopword removal, and bigram detection.

In [ ]:
import nltk
import spacy
import gensim
from nltk.corpus import stopwords

nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

stop_words = set(stopwords.words("english"))
stop_words.update([
    "from", "subject", "re", "edu", "use", "write", "writes", "article",
    "line", "organization", "get", "would", "com", "nntp", "host", "reply",
    "know", "believe", "think", "well", "people", "many", "good",
])


def clean_raw_text(text):
    text = re.sub(r"\S*@\S*\s?", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


docs_clean = [clean_raw_text(doc) for doc in docs_raw]


def preprocess_for_lda(texts, batch_size=500):
    processed = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and len(token.text) > 2
            and token.pos_ in ["NOUN", "ADJ", "VERB", "ADV"]
        ]
        tokens = [token for token in tokens if token not in stop_words]
        processed.append(tokens)
    return processed


data_words = preprocess_for_lda(docs_clean)

bigram = gensim.models.Phrases(data_words, min_count=5, threshold=10)
bigram_mod = gensim.models.phrases.Phraser(bigram)
data_ready = [bigram_mod[doc] for doc in data_words]

print("Example processed document:")
print(data_ready[0][:30])

## Stage 3: Baseline LDA Model

This reproduces and cleans up the baseline from the previous project.

In [ ]:
import gensim.corpora as corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

id2word = corpora.Dictionary(data_ready)
id2word.filter_extremes(no_below=10, no_above=0.5)
corpus = [id2word.doc2bow(text) for text in data_ready]

print(f"Dictionary size: {len(id2word)}")
print(f"Corpus size: {len(corpus)}")

In [ ]:
def compute_coherence_values(dictionary, corpus, texts, topic_nums):
    model_list = []
    coherence_values = []

    for num_topics in topic_nums:
        print(f"Training LDA with {num_topics} topics...")
        model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=RANDOM_STATE,
            passes=5,
            alpha="auto",
        )
        coherence_model = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence="c_v",
        )
        model_list.append(model)
        coherence_values.append(coherence_model.get_coherence())

    return model_list, coherence_values


topic_nums = [5, 8, 10, 11, 14, 17, 20]
lda_models, lda_coherence_scores = compute_coherence_values(
    id2word,
    corpus,
    data_ready,
    topic_nums,
)

coherence_df = pd.DataFrame({
    "num_topics": topic_nums,
    "coherence": lda_coherence_scores,
})

display(coherence_df)

plt.figure(figsize=(8, 5))
sns.lineplot(data=coherence_df, x="num_topics", y="coherence", marker="o")
plt.title("LDA Coherence by Number of Topics")
plt.xlabel("Number of Topics")
plt.ylabel("Coherence Score")
plt.show()

In [ ]:
# Select the best LDA model by coherence.
# You may manually change this if a nearby topic number is easier to interpret.
best_idx = int(np.argmax(lda_coherence_scores))
best_num_topics = topic_nums[best_idx]

print(f"Selected LDA topic number: {best_num_topics}")

lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=best_num_topics,
    random_state=RANDOM_STATE,
    passes=15,
    alpha="auto",
    per_word_topics=True,
)

lda_coherence = CoherenceModel(
    model=lda_model,
    texts=data_ready,
    dictionary=id2word,
    coherence="c_v",
).get_coherence()

print(f"Final LDA coherence: {lda_coherence:.4f}")

for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")

In [ ]:
def get_main_topic(model, corpus):
    topic_data = []
    for doc_id, topic_dist in enumerate(model[corpus]):
        topic_dist = sorted(topic_dist, key=lambda x: x[1], reverse=True)
        topic_id, topic_prob = topic_dist[0]
        topic_data.append([doc_id, topic_id, topic_prob])
    return pd.DataFrame(topic_data, columns=["doc_id", "lda_topic", "lda_topic_probability"])


lda_doc_topics = get_main_topic(lda_model, corpus)
lda_doc_topics["true_label"] = labels
lda_doc_topics["true_category"] = label_names

lda_topics = []
for topic_id in range(best_num_topics):
    words = lda_model.show_topic(topic_id, topn=12)
    lda_topics.append({
        "topic_id": topic_id,
        "keywords": ", ".join([word for word, score in words]),
    })

lda_topics_df = pd.DataFrame(lda_topics)

display(lda_topics_df)
display(lda_doc_topics.head())

## Stage 4: Ground-truth Category Evaluation

The 20 Newsgroups dataset includes real category labels. We use them to evaluate whether unsupervised topics align with known categories.

In [ ]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score


def purity_score(y_true, y_pred):
    contingency = pd.crosstab(y_pred, y_true)
    return np.sum(np.max(contingency.values, axis=1)) / np.sum(contingency.values)


lda_nmi = normalized_mutual_info_score(labels, lda_doc_topics["lda_topic"])
lda_ari = adjusted_rand_score(labels, lda_doc_topics["lda_topic"])
lda_purity = purity_score(labels, lda_doc_topics["lda_topic"])

lda_metrics = pd.DataFrame([{
    "model": "LDA",
    "num_topics": best_num_topics,
    "coherence": lda_coherence,
    "NMI": lda_nmi,
    "ARI": lda_ari,
    "purity": lda_purity,
}])

display(lda_metrics)

In [ ]:
lda_topic_category = pd.crosstab(
    lda_doc_topics["lda_topic"],
    lda_doc_topics["true_category"],
    normalize="index",
)

plt.figure(figsize=(16, 7))
sns.heatmap(lda_topic_category, cmap="YlGnBu")
plt.title("LDA Topic vs. True Newsgroup Category")
plt.xlabel("True Category")
plt.ylabel("LDA Topic")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

## Stage 5: BERTopic Model

BERTopic uses sentence embeddings and clustering to discover semantic topics. This is the main extension beyond the previous LDA project.

For faster testing in Colab, set `USE_SAMPLE = True`. For final results, set it to `False`.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

USE_SAMPLE = False
SAMPLE_SIZE = 5000

if USE_SAMPLE:
    bert_docs = docs_clean[:SAMPLE_SIZE]
    bert_labels = labels[:SAMPLE_SIZE]
    bert_label_names = label_names[:SAMPLE_SIZE]
else:
    bert_docs = docs_clean
    bert_labels = labels
    bert_label_names = label_names

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=25,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

bertopic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=False,
    verbose=True,
)

bertopic_topics, bertopic_probs = bertopic_model.fit_transform(bert_docs)

In [ ]:
bertopic_info = bertopic_model.get_topic_info()
display(bertopic_info.head(20))

bertopic_doc_topics = pd.DataFrame({
    "doc_id": range(len(bert_docs)),
    "bertopic_topic": bertopic_topics,
    "true_label": bert_labels,
    "true_category": bert_label_names,
})

bertopic_topics_rows = []
for topic_id in sorted(set(bertopic_topics)):
    if topic_id == -1:
        keywords = "outlier"
    else:
        words = bertopic_model.get_topic(topic_id) or []
        keywords = ", ".join([word for word, score in words[:12]])
    bertopic_topics_rows.append({
        "topic_id": topic_id,
        "keywords": keywords,
    })

bertopic_topics_df = pd.DataFrame(bertopic_topics_rows)

display(bertopic_topics_df.head(20))
display(bertopic_doc_topics.head())

In [ ]:
valid_mask = bertopic_doc_topics["bertopic_topic"] != -1

bertopic_nmi = normalized_mutual_info_score(
    bertopic_doc_topics.loc[valid_mask, "true_label"],
    bertopic_doc_topics.loc[valid_mask, "bertopic_topic"],
)
bertopic_ari = adjusted_rand_score(
    bertopic_doc_topics.loc[valid_mask, "true_label"],
    bertopic_doc_topics.loc[valid_mask, "bertopic_topic"],
)
bertopic_purity = purity_score(
    bertopic_doc_topics.loc[valid_mask, "true_label"],
    bertopic_doc_topics.loc[valid_mask, "bertopic_topic"],
)

bertopic_metrics = pd.DataFrame([{
    "model": "BERTopic",
    "num_topics": len(set(bertopic_topics)) - (1 if -1 in bertopic_topics else 0),
    "coherence": np.nan,
    "NMI": bertopic_nmi,
    "ARI": bertopic_ari,
    "purity": bertopic_purity,
    "outlier_rate": np.mean(np.array(bertopic_topics) == -1),
}])

display(bertopic_metrics)

In [ ]:
bertopic_topic_category = pd.crosstab(
    bertopic_doc_topics.loc[valid_mask, "bertopic_topic"],
    bertopic_doc_topics.loc[valid_mask, "true_category"],
    normalize="index",
)

plt.figure(figsize=(16, max(8, 0.25 * bertopic_topic_category.shape[0])))
sns.heatmap(bertopic_topic_category, cmap="YlGnBu")
plt.title("BERTopic Topic vs. True Newsgroup Category")
plt.xlabel("True Category")
plt.ylabel("BERTopic Topic")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

## Stage 6: Model Comparison

Use this section to compare LDA and BERTopic quantitatively and qualitatively.

In [ ]:
comparison_metrics = pd.concat([lda_metrics, bertopic_metrics], ignore_index=True)
display(comparison_metrics)

comparison_metrics.to_csv("model_comparison_metrics.csv", index=False)
lda_topics_df.to_csv("lda_topics.csv", index=False)
lda_doc_topics.to_csv("lda_doc_topics.csv", index=False)
bertopic_topics_df.to_csv("bertopic_topics.csv", index=False)
bertopic_doc_topics.to_csv("bertopic_doc_topics.csv", index=False)

print("Saved result files:")
print("- model_comparison_metrics.csv")
print("- lda_topics.csv")
print("- lda_doc_topics.csv")
print("- bertopic_topics.csv")
print("- bertopic_doc_topics.csv")

In [ ]:
# Optional interactive BERTopic visualizations.
# These may be easier to view directly in Colab.

bertopic_model.visualize_barchart(top_n_topics=12)

In [ ]:
bertopic_model.visualize_topics()

## Stage 7: Discussion Template

Fill this part after running all experiments.

### Main Findings

- LDA discovered topics based on word co-occurrence and produced interpretable keyword groups.
- BERTopic discovered topics using semantic embeddings and clustering.
- Based on the category alignment heatmaps, the stronger model appears to be: **[fill in after results]**.
- Based on NMI, ARI, and purity, the stronger model appears to be: **[fill in after results]**.

### Strengths of LDA

- Easier to explain and reproduce.
- Produces a fixed number of topics.
- Works well when topics are strongly separated by vocabulary.

### Weaknesses of LDA

- Depends heavily on preprocessing.
- Uses bag-of-words features and ignores deeper semantic meaning.
- Similar categories may be mixed together.

### Strengths of BERTopic

- Uses sentence embeddings, so it can capture semantic similarity.
- Often creates more natural topic clusters.
- Provides useful interactive visualizations.

### Weaknesses of BERTopic

- More computationally expensive.
- May create many small topics.
- Includes an outlier topic `-1`, which needs interpretation.

### Conclusion

This project extends the previous LDA topic modeling project by adding a transformer-based BERTopic model and evaluating both approaches using true category labels from the 20 Newsgroups dataset. The comparison shows whether modern embedding-based topic modeling improves topic discovery over a traditional probabilistic topic model.